# Python Functions & Descriptors — Advanced Tutorial Lab II

## New problems, incremental experiments, fully worked solutions

This is a **new tutorial**, not a reformatted copy of the earlier 22-problem set. It extends the uploaded lecture's teaching pattern: ask a narrow question → make a prediction → run a small experiment → explain what Python looked up → vary one condition → derive and test a solution.

**Requirements:** Python 3.10+ and a standard Python 3 Jupyter kernel. Everything uses Python's standard library. Run cells **top to bottom**; each problem uses its own clearly named classes to minimize accidental cross-talk. There are no network requests, external files, or non-standard packages.

**How to study:** Hide the solution cell at the end of each problem, predict the results, and then check the assertions. Most code cells are short on purpose: the explanations are part of the solution.

**Scope:** method rebinding, descriptors on metaclasses, error fallback, properties and subclassing, dynamic injection, polymorphic factories, caching, object lifetimes, `__slots__`, static introspection, ABCs, metadata-preserving wrappers, asynchronous methods, generators, dispatch, transactions, and a final integrated design.

## Concept map: what the first lecture established

- A Python function implements `__get__`. Class access normally gives the function; instance access normally gives a **bound method**.
- `bound.__self__` records the receiver; `bound.__func__` points to the underlying function. `types.MethodType(function, instance)` explicitly constructs this kind of binding.
- A function is a **non-data descriptor** (it has `__get__` but not `__set__` or `__delete__`). A data descriptor takes precedence over the instance dictionary, whereas an ordinary function can be shadowed there.
- Attribute lookup has multiple routes: `obj.name`, `Class.name`, and `super().name` need not perform exactly the same descriptor call.

**Important nuance:** An attribute living on a *metaclass* can itself be a descriptor used when accessing a class object. The problems below explore the less obvious consequences of this mechanism.

The lecture's original `MyFunc`/`MethodType` demonstration is our conceptual baseline, but the exercises below introduce different experiments and implementations.

## Notebook conventions and error-checking helper

Several exercises predict an exception. We catch only the expected exception type and assert that it happened, so running the notebook top-to-bottom does **not** stop on a deliberately demonstrated failure.

In [1]:
import asyncio
import functools
import inspect
import types
import weakref
import gc
from abc import ABC, abstractmethod
from decimal import Decimal
from functools import cached_property, lru_cache, singledispatchmethod


def expect_error(exc_type, action):
    """Return the captured expected exception, failing loudly otherwise."""
    try:
        action()
    except exc_type as exc:
        return exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no error was raised")

print("Tools imported; expected-error helper ready")

Tools imported; expected-error helper ready


## Problem 01 — A bound method is a snapshot of binding, not a live lookup

**Your challenge.** Show what happens to an already-captured method when its class function is replaced.

**Predict before running:** Will an old `saved = instance.method` call the new implementation after monkey-patching?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Capture the old method before changing the class

In [2]:
class P01Messenger:
    def greet(self):
        return "original"

p01 = P01Messenger()
p01_saved = p01.greet
print(p01_saved(), p01_saved.__self__ is p01)

original True


**What just happened?** `p01_saved` is a bound method whose `__func__` points to the function that existed when the lookup occurred.

### Step 3 — Replace the attribute and compare both ways of calling

In [3]:
def p01_replacement(self):
    return "replacement"

p01_original_function = p01_saved.__func__
P01Messenger.greet = p01_replacement
print('saved:', p01_saved())
print('fresh:', p01.greet())
print('old function still reachable:', p01_saved.__func__ is p01_original_function)

saved: original
fresh: replacement
old function still reachable: True


**What just happened?** A fresh attribute lookup sees the new class attribute. A bound method already holds references to the old function and the receiver; it is not an indirection through the class dictionary.

### Step 4 — Check what is identical and what is merely equivalent

In [4]:
p01_fresh_a = p01.greet
p01_fresh_b = p01.greet
print('same method object?', p01_fresh_a is p01_fresh_b)
print('same function?', p01_fresh_a.__func__ is p01_fresh_b.__func__)
print('same receiver?', p01_fresh_a.__self__ is p01_fresh_b.__self__)

same method object? False
same function? True
same receiver? True


**What just happened?** Never require repeated method lookups to return the identical method object. Check the stable `__func__` and `__self__` relationships instead.

### Solution 01 — Put the reasoning together

**Answer:** capture-time binding and lookup-time class resolution are different operations. Save a method if you want its current implementation; look it up again if you want a class patch to take effect.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [5]:
assert p01_saved() == 'original'
assert p01.greet() == 'replacement'
assert p01_saved.__func__ is p01_original_function
assert p01.greet.__func__ is p01_replacement
assert p01_saved.__self__ is p01
print('Problem 01: all checks passed')

Problem 01: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 02 — Descriptors on a metaclass control class-object access

**Your challenge.** Implement a read-only computed attribute of a *class*, not of its instances.

**Predict before running:** Which class owns the descriptor that intercepts `Service.report`?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Define a descriptor in the metaclass

In [6]:
class P02ClassLabel:
    def __get__(self, instance, owner):
        return (instance, owner)

class P02Meta(type):
    label = P02ClassLabel()

class P02Service(metaclass=P02Meta):
    pass

p02_access = P02Service.label
print('instance is class object:', p02_access[0] is P02Service)
print('owner is metaclass:', p02_access[1] is P02Meta)

instance is class object: True
owner is metaclass: True


**What just happened?** The class object `P02Service` is an instance of `P02Meta`, so looking up a metaclass descriptor supplies the class object as its `instance` and the metaclass as its `owner`.

### Step 3 — Make the class label useful

In [7]:
class P02ComputedLabel:
    def __get__(self, cls, metaclass):
        if cls is None:
            return self
        return f"{cls.__module__}.{cls.__qualname__}"

class P02ReportingMeta(type):
    report = P02ComputedLabel()

class P02Report(metaclass=P02ReportingMeta):
    pass

print(P02Report.report)

__main__.P02Report


**What just happened?** This class-level computed label belongs to the metaclass lookup path. It does not magically appear as an ordinary instance attribute on `P02Report()` objects.

### Step 4 — Test the contrasting instance lookup

In [8]:
p02_object = P02Report()
print('class has report:', hasattr(P02Report, 'report'))
print('instance has report:', hasattr(p02_object, 'report'))
print('raw descriptor:', P02ReportingMeta.__dict__['report'])

class has report: True
instance has report: False
raw descriptor: <__main__.P02ComputedLabel object at 0x000001AB071DEA50>


**What just happened?** The instance searches its own class MRO, not its class’s metaclass MRO. Also, reading the raw metaclass dictionary does not execute `__get__`.

### Solution 02 — Put the reasoning together

**Answer:** to intercept `SomeClass.name` through descriptor binding, define the descriptor on the metaclass. Ordinary instance access follows a different path.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [9]:
assert P02Service.label == (P02Service, P02Meta)
assert P02Report.report.endswith('.P02Report')
assert not hasattr(P02Report(), 'report')
assert isinstance(P02ReportingMeta.__dict__['report'], P02ComputedLabel)
print('Problem 02: all checks passed')

Problem 02: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 03 — AttributeError inside __get__ triggers __getattr__

**Your challenge.** Observe how an attribute can exist as a descriptor and still invoke fallback logic.

**Predict before running:** Does `__getattr__` run only if the name is absent from the class?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Make a descriptor that fails for missing data

In [10]:
class P03MissingValue:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        raise AttributeError('descriptor has no stored value')

class P03Profile:
    nickname = P03MissingValue()
    def __getattr__(self, name):
        if name == 'nickname':
            return 'anonymous'
        raise AttributeError(name)

p03 = P03Profile()
print('nickname:', p03.nickname)

nickname: anonymous


**What just happened?** The descriptor is found, but its `__get__` raises `AttributeError`; the normal dot-lookup machinery then gives `__getattr__` a chance to provide a value.

### Step 3 — Verify the difference between direct and normal lookup

In [11]:
p03_raw = vars(P03Profile)['nickname']
p03_exc = expect_error(AttributeError, lambda: p03_raw.__get__(p03, P03Profile))
print('direct descriptor error:', str(p03_exc))
print('normal getattr fallback:', getattr(p03, 'nickname'))

direct descriptor error: descriptor has no stored value
normal getattr fallback: anonymous


**What just happened?** Calling `descriptor.__get__` directly does not itself invoke the owner object’s `__getattr__`: the fallback belongs to the surrounding attribute access mechanism.

### Step 4 — Ensure unrelated attributes still fail

In [12]:
p03_missing = expect_error(AttributeError, lambda: p03.nothing_here)
print('unrelated name:', str(p03_missing))

unrelated name: nothing_here


**What just happened?** A robust fallback only handles explicitly supported names; swallowing every `AttributeError` can hide real bugs in nested computations.

### Solution 03 — Put the reasoning together

**Answer:** `__getattr__` is a fallback when normal attribute access raises `AttributeError`, including when a descriptor raises it. Use that behavior intentionally and document it.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [13]:
assert p03.nickname == 'anonymous'
assert p03_raw is P03Profile.nickname
assert isinstance(p03_missing, AttributeError)
print('Problem 03: all checks passed')

Problem 03: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 04 — A subclass can replace an inherited data descriptor

**Your challenge.** Compare a validated property in the base class with an ordinary attribute in a subclass.

**Predict before running:** Does a data descriptor in a parent always outrank an attribute in a child?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Start with a validating base property

In [14]:
class P04Base:
    def __init__(self):
        self._level = 1
    @property
    def level(self):
        return self._level
    @level.setter
    def level(self, value):
        if value < 0:
            raise ValueError('negative level')
        self._level = value

p04_base = P04Base()
expect_error(ValueError, lambda: setattr(p04_base, 'level', -1))
print('base level:', p04_base.level)

base level: 1


**What just happened?** Within one class MRO search, the attribute found first is used. Descriptor *priority* then decides whether that chosen descriptor outranks instance storage.

### Step 3 — Shadow the base property in the subclass namespace

In [15]:
class P04Child(P04Base):
    level = 'unvalidated class value'

p04_child = P04Child()
print('child initial:', p04_child.level)
p04_child.level = -50
print('child instance value:', p04_child.level)
print('base still has property:', isinstance(P04Base.__dict__['level'], property))

child initial: unvalidated class value
child instance value: -50
base still has property: True


**What just happened?** The subclass attribute is found before the parent property. Because this subclass attribute is a plain string, assigning `child.level` creates an instance attribute.

### Step 4 — Contrast explicit use of the parent property

In [16]:
p04_parent_descriptor = P04Base.__dict__['level']
print('base descriptor reads child storage:', p04_parent_descriptor.__get__(p04_child, P04Child))
expect_error(ValueError, lambda: p04_parent_descriptor.__set__(p04_child, -2))

base descriptor reads child storage: 1


ValueError('negative level')

**What just happened?** Explicitly invoking the parent descriptor still performs its validation. This is a demonstration of how lookup selection differs from direct descriptor invocation, not a recommended way to bypass or reinstate a public API.

### Solution 04 — Put the reasoning together

**Answer:** MRO determines the class attribute selected first; descriptor-vs-instance precedence applies *after* that selection. A subclass can replace an inherited property.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [17]:
assert p04_base.level == 1
assert p04_child.level == -50
assert p04_parent_descriptor.__get__(p04_child, P04Child) == 1
assert P04Child.__dict__['level'] == 'unvalidated class value'
print('Problem 04: all checks passed')

Problem 04: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 05 — Extend just the setter of an inherited property

**Your challenge.** Create a subclass setter that adds checks while reusing the original getter.

**Predict before running:** What object does `@Base.value.setter` build, and which class does it modify?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Define the original property

In [18]:
class P05Account:
    def __init__(self, balance):
        self.balance = balance
    @property
    def balance(self):
        return self._balance
    @balance.setter
    def balance(self, amount):
        if amount < 0:
            raise ValueError('negative balance')
        self._balance = amount

print('base property:', isinstance(P05Account.__dict__['balance'], property))

base property: True


**What just happened?** A property is a descriptor containing accessor functions; `.setter(new_function)` returns a new property with the same getter and a different setter.

### Step 3 — Create a stricter property in the subclass

In [19]:
class P05CappedAccount(P05Account):
    @P05Account.balance.setter
    def balance(self, amount):
        if amount > 100:
            raise ValueError('exceeds cap')
        P05Account.balance.fset(self, amount)

p05 = P05CappedAccount(25)
print('inherited getter:', p05.balance)
print('same getter:', P05Account.balance.fget is P05CappedAccount.balance.fget)

inherited getter: 25
same getter: True


**What just happened?** The base property is left unchanged. Calling the base setter explicitly inside the subclass setter retains the original non-negative check.

### Step 4 — Test both validation boundaries

In [20]:
p05.balance = 100
print('at cap:', p05.balance)
print('negative:', type(expect_error(ValueError, lambda: setattr(p05, 'balance', -1))).__name__)
print('above cap:', type(expect_error(ValueError, lambda: setattr(p05, 'balance', 101))).__name__)

at cap: 100
negative: ValueError
above cap: ValueError


**What just happened?** Property reuse here is explicit. There is no implicit super-call from the subclass setter to the base setter.

### Solution 05 — Put the reasoning together

**Answer:** use `@Base.property_name.setter` to make a distinct subclass property that reuses its getter. Invoke the base setter deliberately if you want inherited validation as well.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [21]:
assert p05.balance == 100
assert P05Account(150).balance == 150
assert P05Account.__dict__['balance'] is not P05CappedAccount.__dict__['balance']
assert P05Account.balance.fget is P05CappedAccount.balance.fget
print('Problem 05: all checks passed')

Problem 05: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 06 — Inject an instance-specific method without changing siblings

**Your challenge.** Attach an independently bound behavior to only one object.

**Predict before running:** If you assign a plain function directly into `instance.__dict__`, is it automatically bound?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Assign a plain function to an instance

In [22]:
class P06Robot:
    def __init__(self, name):
        self.name = name

def p06_status(self, marker='ok'):
    return f'{self.name}:{marker}'

p06_a = P06Robot('A')
p06_b = P06Robot('B')
p06_a.status = p06_status
print('raw object is same function:', p06_a.status is p06_status)
print('calling without self fails:', type(expect_error(TypeError, lambda: p06_a.status())).__name__)

raw object is same function: True
calling without self fails: TypeError


**What just happened?** The function stored directly on the *instance* is returned as-is. Python only applies its ordinary method-descriptor binding when it is found on a class.

### Step 3 — Bind explicitly and avoid modifying other instances

In [23]:
p06_a.status = types.MethodType(p06_status, p06_a)
print(p06_a.status())
print('B has status?', hasattr(p06_b, 'status'))
print('explicit receiver:', p06_a.status.__self__ is p06_a)

A:ok
B has status? False
explicit receiver: True


**What just happened?** `types.MethodType` makes a bound method. Instance A gets a normal callable accepting no explicit `self`, while instance B is untouched.

### Step 4 — Contrast a class patch with instance patch

In [24]:
P06Robot.status = p06_status
print('B now:', p06_b.status('ready'))
print('A still uses instance method:', p06_a.status('busy'))
print('A instance override:', p06_a.__dict__['status'].__self__ is p06_a)

B now: B:ready
A still uses instance method: A:busy
A instance override: True


**What just happened?** The class function is a non-data descriptor, so A’s instance attribute shadows it; B retrieves it from the class and gets automatic binding.

### Solution 06 — Put the reasoning together

**Answer:** functions in class dictionaries bind automatically; functions in instance dictionaries do not. `MethodType` is appropriate when a one-off bound method is intentional.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [25]:
assert p06_a.status() == 'A:ok'
assert p06_b.status() == 'B:ok'
assert p06_b.status.__func__ is p06_status
assert p06_a.status.__self__ is p06_a
print('Problem 06: all checks passed')

Problem 06: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 07 — An inherited classmethod makes a polymorphic factory

**Your challenge.** Write one alternative constructor that returns the class through which it was invoked.

**Predict before running:** When called on a subclass, does an inherited `classmethod` receive the base class or the subclass?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Write a factory that uses cls, not a hard-coded class

In [26]:
class P07Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    @classmethod
    def from_csv(cls, record):
        parts = record.split(',')
        if len(parts) != 2:
            raise ValueError('exactly two values required')
        return cls(*(int(value.strip()) for value in parts))

p07_base = P07Point.from_csv('3, 4')
print(type(p07_base).__name__, (p07_base.x, p07_base.y))

P07Point (3, 4)


**What just happened?** A classmethod’s bound receiver is the class object. `cls(...)` uses that receiver and can therefore construct subclasses.

### Step 3 — Inherit the constructor unchanged

In [27]:
class P07ColoredPoint(P07Point):
    def __init__(self, x, y, color='blue'):
        super().__init__(x, y)
        self.color = color

p07_child = P07ColoredPoint.from_csv('5, 6')
print(type(p07_child).__name__, p07_child.color)
print('bound receiver:', P07ColoredPoint.from_csv.__self__ is P07ColoredPoint)

P07ColoredPoint blue
bound receiver: True


**What just happened?** The inherited descriptor binds to the subclass being accessed. This only works when the subclass constructor accepts the arguments provided by the factory.

### Step 4 — Show why hard-coding the base name breaks polymorphism

In [28]:
class P07RigidPoint(P07Point):
    @classmethod
    def from_csv(cls, record):
        x, y = (int(item) for item in record.split(','))
        return P07RigidPoint(x, y)

class P07RigidChild(P07RigidPoint):
    pass

p07_rigid = P07RigidChild.from_csv('1,2')
print('actual:', type(p07_rigid).__name__, 'expected subclass:', P07RigidChild.__name__)

actual: P07RigidPoint expected subclass: P07RigidChild


**What just happened?** A factory using the literal base-class name ignores the class bound as `cls` and can silently produce the wrong type.

### Solution 07 — Put the reasoning together

**Answer:** a classmethod receives the class through which the method is resolved; a polymorphic factory constructs through `cls`. Validate parsing and constructor compatibility separately.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [29]:
assert type(p07_base) is P07Point
assert type(p07_child) is P07ColoredPoint
assert p07_child.x == 5 and p07_child.y == 6
assert type(p07_rigid) is P07RigidPoint
expect_error(ValueError, lambda: P07Point.from_csv('one value'))
print('Problem 07: all checks passed')

Problem 07: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 08 — cached_property is a one-time computation until invalidated

**Your challenge.** Measure computation counts and deliberately invalidate one instance.

**Predict before running:** Does a cached property recompute on every access, and where is its result kept?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Create a property with observable computation

In [30]:
class P08Invoice:
    def __init__(self, prices):
        self.prices = list(prices)
        self.computations = 0

    @cached_property
    def total(self):
        self.computations += 1
        return sum(self.prices)

p08_a = P08Invoice([2, 3])
print(p08_a.total, p08_a.total, p08_a.computations)
print('stored value:', p08_a.__dict__['total'])

5 5 1
stored value: 5


**What just happened?** `cached_property` initially behaves like a non-data descriptor. It writes the result to the instance dictionary under its attribute name; later reads find the instance entry instead.

### Step 3 — Show that mutating dependencies does not auto-invalidate

In [31]:
p08_a.prices.append(10)
print('stale total:', p08_a.total)
del p08_a.total
print('fresh total after deletion:', p08_a.total)
print('computations:', p08_a.computations)

stale total: 5
fresh total after deletion: 15
computations: 2


**What just happened?** `cached_property` does not discover data dependencies. Deleting the cached instance attribute lets the descriptor execute again on the next read.

### Step 4 — Confirm invalidation is local to each instance

In [32]:
p08_b = P08Invoice([20])
_ = p08_b.total
p08_a.prices.append(1)
del p08_a.total
print('A:', p08_a.total, 'B:', p08_b.total)
print('counts:', p08_a.computations, p08_b.computations)

A: 16 B: 20
counts: 3 1


**What just happened?** Each instance holds its own cache entry. In concurrent code, do not assume the function executes exactly once without providing suitable synchronization yourself.

### Solution 08 — Put the reasoning together

**Answer:** instance dictionaries store cached values. Invalidate explicitly after relevant mutations; a property decorator alone cannot track dependencies.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [33]:
assert p08_a.total == 16
assert p08_a.computations == 3
assert p08_b.total == 20 and p08_b.computations == 1
assert p08_a.__dict__['total'] == 16
print('Problem 08: all checks passed')

Problem 08: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 09 — Why cached_property fails on dict-less slotted objects

**Your challenge.** Reproduce a standard-library cache limitation, then repair the class layout.

**Predict before running:** Will `cached_property` work if a class provides only `__slots__` and no instance dictionary?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Make the smallest slotted object

In [34]:
class P09Compact:
    __slots__ = ('x',)
    def __init__(self, x):
        self.x = x
    @cached_property
    def doubled(self):
        return 2 * self.x

p09_compact = P09Compact(7)
p09_error = expect_error(TypeError, lambda: p09_compact.doubled)
print('failure:', str(p09_error))

failure: No '__dict__' attribute on 'P09Compact' instance to cache 'doubled' property.


**What just happened?** `cached_property` needs a mutable per-instance `__dict__` in which to save the computed result. A class declaring only `__slots__ = ("x",)` does not provide one.

### Step 3 — Give the slotted class an explicit instance dictionary

In [35]:
class P09CacheReady:
    __slots__ = ('x', '__dict__')
    def __init__(self, x):
        self.x = x
    @cached_property
    def doubled(self):
        return 2 * self.x

p09_ready = P09CacheReady(7)
print('first:', p09_ready.doubled)
print('cache contents:', p09_ready.__dict__)

first: 14
cache contents: {'doubled': 14}


**What just happened?** Adding `__dict__` to the slots allows arbitrary instance attributes, including the cached value. That partially sacrifices the memory constraints that slots are often used to impose.

### Step 4 — Invalidate after changing an input

In [36]:
p09_ready.x = 9
print('before invalidation:', p09_ready.doubled)
del p09_ready.doubled
print('after invalidation:', p09_ready.doubled)

before invalidation: 14
after invalidation: 18


**What just happened?** The slotted `x` value changes, but the ordinary cached value in `__dict__` remains until it is deleted.

### Solution 09 — Put the reasoning together

**Answer:** `cached_property` is not a cache for arbitrary slot-only objects. Give instances a dictionary, or design a different storage policy appropriate for your requirements.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [37]:
assert not hasattr(p09_compact, '__dict__')
assert p09_ready.__dict__['doubled'] == 18
assert p09_ready.doubled == 18
assert isinstance(p09_error, TypeError)
print('Problem 09: all checks passed')

Problem 09: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 10 — A cached method can keep self alive

**Your challenge.** Demonstrate a lifetime consequence of decorating an instance method with `lru_cache`.

**Predict before running:** What does a shared cache key contain when a method is called as `object.method(arg)`?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Decorate the method and retain only a weak reference

In [38]:
class P10Calculator:
    @lru_cache(maxsize=None)
    def square(self, number):
        return number * number

p10_item = P10Calculator()
p10_weak = weakref.ref(p10_item)
print('result:', p10_item.square(6))
print('cache:', P10Calculator.square.cache_info())

result: 36
cache: CacheInfo(hits=0, misses=1, maxsize=None, currsize=1)


**What just happened?** The shared cache is attached to the function wrapper on the class. Calls include `self` in the cache key, so cached entries can retain instances while entries remain in the cache.

### Step 3 — Delete the external reference and inspect object lifetime

In [39]:
del p10_item
gc.collect()
print('still retained by cache:', p10_weak() is not None)

still retained by cache: True


**What just happened?** This demonstration assumes the default identity-hashable instance and a cache entry that has not been evicted. The remaining strong reference is held by the shared cache key.

### Step 4 — Clear the shared cache and verify collection

In [40]:
P10Calculator.square.cache_clear()
gc.collect()
print('released after clear:', p10_weak() is None)

released after clear: True


**What just happened?** Cache clearing removes the key and frees the instance once no other strong references remain. This is one reason method-level `lru_cache` is not automatically a per-instance cache.

### Solution 10 — Put the reasoning together

**Answer:** an `lru_cache` on an instance method is normally shared at the class-wrapper level, and its keys include `self`. Pick a bounded cache, design per-instance storage, or clear caches according to the object-lifetime requirements.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [41]:
assert p10_weak() is None
assert P10Calculator.square.cache_info().currsize == 0
p10_another = P10Calculator()
assert p10_another.square(8) == 64
P10Calculator.square.cache_clear()
print('Problem 10: all checks passed')

Problem 10: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 11 — A stored bound method retains its receiver

**Your challenge.** Contrast an ordinary strong bound-method reference with `weakref.WeakMethod`.

**Predict before running:** If the sole surviving variable is a bound method, can its receiver be garbage-collected?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Capture a strong bound method

In [42]:
class P11Worker:
    def run(self):
        return 'done'

p11_worker = P11Worker()
p11_reference = weakref.ref(p11_worker)
p11_strong_method = p11_worker.run
del p11_worker
gc.collect()
print('receiver alive:', p11_reference() is not None)
print('call still works:', p11_strong_method())

receiver alive: True
call still works: done


**What just happened?** A bound method holds its receiver in `__self__`. Deleting the original variable does not free that object while another bound-method reference survives.

### Step 3 — Release the strong reference and confirm collection

In [43]:
del p11_strong_method
gc.collect()
print('receiver gone:', p11_reference() is None)

receiver gone: True


**What just happened?** The weak reference allows observation without keeping the object alive itself. Garbage collection is invoked explicitly to make this lesson deterministic for the notebook kernel.

### Step 4 — Use WeakMethod for a callback registry

In [44]:
p11_worker2 = P11Worker()
p11_weak_method = weakref.WeakMethod(p11_worker2.run)
print('before deletion:', p11_weak_method()())
del p11_worker2
gc.collect()
print('after deletion:', p11_weak_method())

before deletion: done
after deletion: None


**What just happened?** `WeakMethod` stores enough information to reconstruct a bound method while both parts are live, without keeping the receiver alive. Check for `None` before calling it.

### Solution 11 — Put the reasoning together

**Answer:** a bound method is a *strong* ownership path to `__self__`; `WeakMethod` is the standard-library tool for a callback reference that should not extend receiver lifetime.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [45]:
assert p11_reference() is None
assert p11_weak_method() is None
print('Problem 11: all checks passed')

Problem 11: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 12 — Use WeakKeyDictionary storage with slot-only objects

**Your challenge.** Implement an externally stored data descriptor that works when instances have no `__dict__`.

**Predict before running:** What additional slot must support weak references to instances?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Define a weakly keyed descriptor with explicit missing-value behavior

In [46]:
class P12ExternalValue:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()
    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError('value has not been assigned') from None
    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError('integer required')
        self.values[instance] = value
    def __delete__(self, instance):
        del self.values[instance]

**What just happened?** This is a data descriptor. Its storage belongs to the descriptor, not to the instance. Weak keys avoid extending the lifetime of an instance merely because it has a descriptor value.

### Step 3 — Allow weak references in a slotted class

In [47]:
class P12Slotted:
    __slots__ = ('__weakref__',)
    score = P12ExternalValue()

p12_record = P12Slotted()
p12_record.score = 40
print('score:', p12_record.score)
print('instance dictionary?', hasattr(p12_record, '__dict__'))
print('external entries:', len(P12Slotted.score.values))

score: 40
instance dictionary? False
external entries: 1


**What just happened?** `__weakref__` enables weak references for instances that otherwise have no base-provided weak-reference support. The default identity equality and hashing are suitable here; custom unhashable instances need a different storage design.

### Step 4 — Test validation, deletion, and weak-key cleanup

In [48]:
expect_error(TypeError, lambda: setattr(p12_record, 'score', 'bad'))
assert p12_record.score == 40
p12_record.score = 41
del p12_record.score
expect_error(AttributeError, lambda: p12_record.score)
p12_record.score = 42
p12_weak = weakref.ref(p12_record)
del p12_record
gc.collect()
print('instance collected:', p12_weak() is None)
print('remaining keys:', len(P12Slotted.score.values))

instance collected: True
remaining keys: 0


**What just happened?** The weakly keyed mapping drops the entry after its key is collected. Beware: if the *value* holds a strong reference back to the key, a WeakKeyDictionary alone cannot break that reference cycle.

### Solution 12 — Put the reasoning together

**Answer:** external weak-key storage can support slotted objects without `__dict__`, provided keys are weak-referenceable and hashable. Validation occurs before mutation.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [49]:
assert p12_weak() is None
assert len(P12Slotted.score.values) == 0
assert not hasattr(P12Slotted(), '__dict__')
print('Problem 12: all checks passed')

Problem 12: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 13 — Read attributes without executing descriptor code

**Your challenge.** Inspect a potentially expensive or side-effectful descriptor safely.

**Predict before running:** Does `getattr` reveal the descriptor object or execute its `__get__`?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Write a descriptor with visible side effects

In [50]:
class P13Dangerous:
    def __init__(self):
        self.calls = 0
    def __get__(self, instance, owner):
        self.calls += 1
        return 'computed'

class P13Container:
    item = P13Dangerous()

p13 = P13Container()
p13_descriptor = vars(P13Container)['item']
print('initial calls:', p13_descriptor.calls)
print('getattr:', getattr(p13, 'item'))
print('calls after:', p13_descriptor.calls)

initial calls: 0
getattr: computed
calls after: 1


**What just happened?** Ordinary `getattr` performs attribute lookup, which invokes the descriptor. This can do I/O, logging, validation, or arbitrary user code.

### Step 3 — Use inspect.getattr_static for a raw view

In [51]:
p13_static_instance = inspect.getattr_static(p13, 'item')
p13_static_class = inspect.getattr_static(P13Container, 'item')
print('raw object on instance:', p13_static_instance is p13_descriptor)
print('raw object on class:', p13_static_class is p13_descriptor)
print('calls unchanged:', p13_descriptor.calls)

raw object on instance: True
raw object on class: True
calls unchanged: 1


**What just happened?** `inspect.getattr_static` avoids dynamic lookup, including executing `__get__`, `__getattr__`, and custom `__getattribute__`. Its result can be a raw descriptor that needs interpretation.

### Step 4 — Use static inspection on an ordinary method as well

In [52]:
class P13Regular:
    def method(self, value):
        return value + 1

p13_regular = P13Regular()
p13_raw_function = inspect.getattr_static(p13_regular, 'method')
p13_bound = getattr(p13_regular, 'method')
print('static is function:', inspect.isfunction(p13_raw_function))
print('dynamic is method:', inspect.ismethod(p13_bound))

static is function: True
dynamic is method: True


**What just happened?** Static inspection is a diagnostic technique; it intentionally does not produce the same user-facing behavior as normal attribute access.

### Solution 13 — Put the reasoning together

**Answer:** use static inspection when examining untrusted or side-effectful attributes; use ordinary `getattr` when you actually want normal attribute semantics.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [53]:
assert p13_descriptor.calls == 1
assert p13_static_instance is p13_descriptor
assert p13_raw_function is P13Regular.__dict__['method']
assert p13_bound.__func__ is p13_raw_function
print('Problem 13: all checks passed')

Problem 13: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 14 — Make a custom descriptor cooperate with ABCMeta

**Your challenge.** Define a custom descriptor that can be marked abstract and later implemented.

**Predict before running:** How does an abstract base class recognize a descriptor as abstract?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Publish abstractness on the descriptor object

In [54]:
class P14MethodDescriptor:
    def __init__(self, function):
        self.function = function
        self.__isabstractmethod__ = bool(getattr(function, '__isabstractmethod__', False))
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return types.MethodType(self.function, instance)

class P14Base(ABC):
    @P14MethodDescriptor
    @abstractmethod
    def describe(self):
        raise NotImplementedError

print('abstract names:', P14Base.__abstractmethods__)

abstract names: frozenset({'describe'})


**What just happened?** `ABCMeta` relies on `__isabstractmethod__` when determining whether class attributes fulfill abstract contracts. Our wrapper must expose the abstractness of the wrapped function.

### Step 3 — Prove an incomplete subclass cannot be instantiated

In [55]:
class P14Incomplete(P14Base):
    pass

p14_failure = expect_error(TypeError, P14Incomplete)
print('error type:', type(p14_failure).__name__)
print('still abstract:', inspect.isabstract(P14Incomplete))

error type: TypeError
still abstract: True


**What just happened?** The inherited abstract descriptor remains abstract, so instantiation is rejected even though the attribute exists.

### Step 4 — Implement the name and regain concrete behavior

In [56]:
class P14Complete(P14Base):
    def describe(self):
        return 'implemented'

p14_concrete = P14Complete()
print('concrete:', p14_concrete.describe())
print('abstract names:', P14Complete.__abstractmethods__)

concrete: implemented
abstract names: frozenset()


**What just happened?** An overriding concrete method fulfills the abstract requirement. Decorator order matters: `@abstractmethod` is applied to the function first, and our descriptor wrapper receives it afterward.

### Solution 14 — Put the reasoning together

**Answer:** custom abstract descriptors should advertise the wrapped function’s `__isabstractmethod__` status. This keeps normal ABC enforcement intact.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [57]:
assert 'describe' in P14Base.__abstractmethods__
assert 'describe' in P14Incomplete.__abstractmethods__
assert P14Complete.__abstractmethods__ == frozenset()
assert p14_concrete.describe() == 'implemented'
print('Problem 14: all checks passed')

Problem 14: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 15 — Build a tracing method descriptor without losing signatures

**Your challenge.** Wrap an ordinary method, log calls, and preserve metadata for `inspect.signature`.

**Predict before running:** What happens if a descriptor returns a generic `*args, **kwargs` wrapper without `functools.wraps`?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Construct a wrapper once, then bind it during access

In [58]:
class P15TraceMethod:
    def __init__(self, function):
        self.function = function
        self.events = []

        @functools.wraps(function)
        def wrapper(receiver, *args, **kwargs):
            result = function(receiver, *args, **kwargs)
            self.events.append((receiver, args, kwargs, result))
            return result
        self.wrapper = wrapper

    def __get__(self, instance, owner):
        if instance is None:
            return self.wrapper
        return types.MethodType(self.wrapper, instance)

**What just happened?** `functools.wraps` supplies `__wrapped__`, the original name, and docstring, so introspection follows the original function. A new method is bound only on instance access.

### Step 3 — Decorate and compare signatures

In [59]:
class P15Math:
    @P15TraceMethod
    def multiply(self, left: int, right: int = 2) -> int:
        """Multiply the supplied values."""
        return left * right

p15_math = P15Math()
print('class signature:', inspect.signature(P15Math.multiply))
print('bound signature:', inspect.signature(p15_math.multiply))
print('docstring:', p15_math.multiply.__doc__)

class signature: (self, left: int, right: int = 2) -> int
bound signature: (left: int, right: int = 2) -> int
docstring: Multiply the supplied values.


**What just happened?** The class view reports the explicit receiver parameter, while the bound-method view hides the already-supplied receiver. The wrapper retains documentation and annotations.

### Step 4 — Invoke the method and inspect the raw descriptor log

In [60]:
print('result:', p15_math.multiply(3, right=4))
p15_descriptor = vars(P15Math)['multiply']
print('events:', len(p15_descriptor.events))
print('arguments:', p15_descriptor.events[-1][1:3])
print('original preserved:', P15Math.multiply.__wrapped__ is p15_descriptor.function)

result: 12
events: 1
arguments: ((3,), {'right': 4})
original preserved: True


**What just happened?** Class attribute access returns the wrapper here; to inspect the descriptor’s own log we use `vars(P15Math)`. Unlike a production tracing library, this teaching example stores its log at class-descriptor scope.

### Solution 15 — Put the reasoning together

**Answer:** construct the wrapper with `functools.wraps`, return it for class access, and bind it with `MethodType` for instance access. Keep the logging scope and resource costs explicit.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [61]:
assert p15_math.multiply(5) == 10
assert len(p15_descriptor.events) == 2
assert list(inspect.signature(p15_math.multiply).parameters) == ['left', 'right']
assert P15Math.multiply.__name__ == 'multiply'
assert P15Math.multiply.__wrapped__ is p15_descriptor.function
print('Problem 15: all checks passed')

Problem 15: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 16 — Trace async methods without accidentally returning a coroutine object

**Your challenge.** Create an async-aware descriptor that awaits the wrapped operation.

**Predict before running:** Should the wrapper itself be `def` or `async def` if introspection must recognize it as a coroutine function?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Build an async wrapper with normal descriptor binding

In [62]:
class P16AsyncTrace:
    def __init__(self, function):
        if not inspect.iscoroutinefunction(function):
            raise TypeError('async function required')
        self.events = []

        @functools.wraps(function)
        async def wrapper(receiver, *args, **kwargs):
            self.events.append('started')
            try:
                result = await function(receiver, *args, **kwargs)
            except Exception:
                self.events.append('failed')
                raise
            self.events.append('finished')
            return result
        self.wrapper = wrapper

    def __get__(self, instance, owner):
        if instance is None:
            return self.wrapper
        return types.MethodType(self.wrapper, instance)

**What just happened?** The wrapper is `async def` and awaits the original coroutine. The `try/except` observes failures and re-raises them without turning errors into success values.

### Step 3 — Call and await in a notebook cell

In [63]:
class P16Fetcher:
    @P16AsyncTrace
    async def fetch(self, item):
        await asyncio.sleep(0)
        return item.upper()

p16_fetcher = P16Fetcher()
print('recognized coroutine function:', inspect.iscoroutinefunction(p16_fetcher.fetch))
p16_result = await p16_fetcher.fetch('hello')
print('result:', p16_result)

recognized coroutine function: True
result: HELLO


**What just happened?** Jupyter supports top-level `await`, so we use it instead of `asyncio.run()`; the latter can fail when called inside an already-running notebook event loop.

### Step 4 — Handle a failing async method without losing the exception

In [64]:
class P16Failer:
    @P16AsyncTrace
    async def fail(self):
        raise LookupError('not found')

p16_failer = P16Failer()
try:
    await p16_failer.fail()
except LookupError as exc:
    print('preserved exception:', str(exc))
else:
    raise AssertionError('failure should propagate')
print('failure trace:', vars(P16Failer)['fail'].events)

preserved exception: not found
failure trace: ['started', 'failed']


**What just happened?** The tracing descriptor records failure and preserves the original exception. A plain synchronous wrapper would not offer the same direct coroutine-function introspection.

### Solution 16 — Put the reasoning together

**Answer:** maintain async function identity by wrapping with `async def`, await the delegated coroutine, and propagate errors. The bound method and the class function remain inspectable.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [65]:
assert p16_result == 'HELLO'
assert inspect.iscoroutinefunction(P16Fetcher.fetch)
assert vars(P16Fetcher)['fetch'].events == ['started', 'finished']
assert vars(P16Failer)['fail'].events == ['started', 'failed']
print('Problem 16: all checks passed')

Problem 16: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 17 — A generator-method wrapper must preserve lazy execution

**Your challenge.** Write a descriptor that traces iteration without consuming the whole generator.

**Predict before running:** When does code inside a `def ...: yield ...` wrapper begin running: method call or first iteration?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Return a real generator function from the descriptor

In [66]:
class P17StreamTrace:
    def __init__(self, function):
        if not inspect.isgeneratorfunction(function):
            raise TypeError('generator function required')
        self.events = []

        @functools.wraps(function)
        def wrapper(receiver, *args, **kwargs):
            self.events.append('started iteration')
            try:
                yield from function(receiver, *args, **kwargs)
            finally:
                self.events.append('iteration exited')
        self.wrapper = wrapper

    def __get__(self, instance, owner):
        if instance is None:
            return self.wrapper
        return types.MethodType(self.wrapper, instance)

**What just happened?** Because the wrapper itself contains `yield from`, calling it makes a generator without executing the body. The `finally` runs when execution completes or the generator is explicitly closed after it has started.

### Step 3 — Observe call-time versus iteration-time effects

In [67]:
class P17Sequence:
    @P17StreamTrace
    def numbers(self, count):
        for value in range(count):
            yield value * 2

p17_sequence = P17Sequence()
p17_stream = p17_sequence.numbers(3)
p17_descriptor = vars(P17Sequence)['numbers']
print('events immediately:', p17_descriptor.events)
print('first value:', next(p17_stream))
print('events after first value:', p17_descriptor.events)

events immediately: []
first value: 0
events after first value: ['started iteration']


**What just happened?** The trace is empty right after the call; the first `next()` starts the wrapper and the delegated generator. No eager `list(...)` conversion occurs.

### Step 4 — Finish the iteration and handle early close

In [68]:
print('remaining values:', list(p17_stream))
print('events after completion:', p17_descriptor.events)
p17_other = p17_sequence.numbers(100)
print('early first value:', next(p17_other))
p17_other.close()
print('events after close:', p17_descriptor.events)

remaining values: [2, 4]
events after completion: ['started iteration', 'iteration exited']
early first value: 0
events after close: ['started iteration', 'iteration exited', 'started iteration', 'iteration exited']


**What just happened?** Closing a started generator causes its `finally` cleanup to execute. An unstarted generator may be closed without entering the wrapper body; do not confuse creating a generator with beginning work.

### Solution 17 — Put the reasoning together

**Answer:** preserve streaming by making the wrapper itself a generator and using `yield from`. Logging and cleanup should reflect actual iteration rather than mere generator creation.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [69]:
assert inspect.isgeneratorfunction(P17Sequence.numbers)
assert p17_descriptor.events == [
    'started iteration', 'iteration exited',
    'started iteration', 'iteration exited',
]
assert list(p17_sequence.numbers(2)) == [0, 2]
assert p17_descriptor.events[-2:] == ['started iteration', 'iteration exited']
print('Problem 17: all checks passed')

Problem 17: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 18 — Single-dispatch instance methods are descriptors too

**Your challenge.** Implement method overloads based on the first argument after `self`.

**Predict before running:** Which handler is selected for `bool`, given that `bool` subclasses `int`?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Define a base handler and register more specific implementations

In [70]:
class P18Formatter:
    @singledispatchmethod
    def format(self, value):
        return f'generic:{type(value).__name__}'

    @format.register
    def _(self, value: int):
        return f'integer:{value}'

    @format.register
    def _(self, value: str):
        return f'text:{value.upper()}'

p18 = P18Formatter()
print(p18.format(5))
print(p18.format('hi'))
print(p18.format([1, 2]))

integer:5
text:HI
generic:list


**What just happened?** `singledispatchmethod` is a descriptor that dispatches on the first non-`self` argument. The fallback implementation handles unregistered types.

### Step 3 — Expose the inheritance effect of bool

In [71]:
print('bool is int subclass:', issubclass(bool, int))
print('true dispatch before registering bool:', p18.format(True))

bool is int subclass: True
true dispatch before registering bool: integer:True


**What just happened?** Without a specific `bool` registration, dispatch chooses the registered `int` implementation through the type hierarchy. This is type-based, not value-based, dispatch.

### Step 4 — Add an exact bool handler through the raw descriptor

In [72]:
p18_descriptor = vars(P18Formatter)['format']

@p18_descriptor.register(bool)
def p18_format_bool(self, value):
    return 'boolean:YES' if value else 'boolean:NO'

print('now bool:', p18.format(True))
print('int unchanged:', p18.format(1))

now bool: boolean:YES
int unchanged: integer:1


**What just happened?** Registration changes the dispatch table shared by this class descriptor. The newly registered exact `bool` handler takes priority over the `int` handler.

### Solution 18 — Put the reasoning together

**Answer:** `singledispatchmethod` combines method binding with type-based dispatch, inspecting the first non-receiver argument. Registration is shared on the descriptor, not per instance.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [73]:
assert p18.format(True) == 'boolean:YES'
assert p18.format(False) == 'boolean:NO'
assert p18.format(2) == 'integer:2'
assert p18.format('ok') == 'text:OK'
assert p18.format(None) == 'generic:NoneType'
print('Problem 18: all checks passed')

Problem 18: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 19 — Reject invalid descriptor writes before committing state

**Your challenge.** Build a strict integer field with an audit trail and no partial mutation after validation fails.

**Predict before running:** If a setter validates *after* writing, what goes wrong when validation raises?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Name the storage slot with __set_name__

In [74]:
class P19AtomicInteger:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = '_' + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.private_name]
        except KeyError:
            raise AttributeError(self.public_name + ' is not initialized') from None

    def __set__(self, instance, value):
        if type(value) is not int:  # deliberately reject bool
            raise TypeError(self.public_name + ' must be an integer')
        if not 0 <= value <= 10:
            raise ValueError(self.public_name + ' must be in [0, 10]')
        old = instance.__dict__.get(self.private_name)
        instance.__dict__[self.private_name] = value
        instance.history.append((self.public_name, old, value))

    def __delete__(self, instance):
        raise AttributeError('deleting required fields is forbidden')

**What just happened?** All validation occurs **before** the write. `__set_name__` records the public name and a separate storage key. We assume `history` is an ordinary list and that its append succeeds; this exercise demonstrates atomicity with respect to *validation failures*, not arbitrary exceptions during logging.

### Step 3 — Build a client class and initialize dependencies first

In [75]:
class P19Ticket:
    priority = P19AtomicInteger()
    def __init__(self, priority):
        self.history = []
        self.priority = priority

p19_ticket = P19Ticket(4)
print('priority:', p19_ticket.priority)
print('audit:', p19_ticket.history)
print('private storage:', p19_ticket.__dict__['_priority'])

priority: 4
audit: [('priority', None, 4)]
private storage: 4


**What just happened?** Initializing `history` first matters because the descriptor’s `__set__` uses it. Data descriptors coordinate initialization with the owning class; they do not control construction order.

### Step 4 — Attempt invalid writes and compare snapshots

In [76]:
p19_before = dict(p19_ticket.__dict__)
expect_error(TypeError, lambda: setattr(p19_ticket, 'priority', True))
expect_error(ValueError, lambda: setattr(p19_ticket, 'priority', 99))
print('priority unchanged:', p19_ticket.priority)
print('history unchanged:', p19_ticket.history == p19_before['history'])
print('deleting fails:', type(expect_error(AttributeError, lambda: delattr(p19_ticket, 'priority'))).__name__)

priority unchanged: 4
history unchanged: True
deleting fails: AttributeError


**What just happened?** The failed validation paths did not touch the state. The shallow copy of the dictionary is enough here because these failed operations do not mutate the history list either.

### Solution 19 — Put the reasoning together

**Answer:** validate first and only then write or record the change. A read/write/delete policy should be explicit; adding `__delete__` also makes deletion behavior easy to reason about.

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [77]:
assert p19_ticket.priority == 4
assert p19_ticket.history == [('priority', None, 4)]
p19_ticket.priority = 10
assert p19_ticket.history[-1] == ('priority', 4, 10)
assert P19Ticket.priority.public_name == 'priority'
print('Problem 19: all checks passed')

Problem 19: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

## Problem 20 — Capstone: observable fields with dependency-aware cache invalidation

**Your challenge.** Create a small, consistent cart model using two validating descriptors and a cached derived total.

**Predict before running:** Can a failed field assignment leave both the cached total and the event log exactly unchanged?

### Step 1 — Build the smallest experiment

Start with a few objects and explicitly name the attribute being looked up. Run the next cell and compare the output with your prediction. Later cells add one complication at a time.

### Step 2 — Translate the problem into invariants

Our cart must satisfy all of the following:

1. `price` is a finite, nonnegative `Decimal`, normalized to two decimal places; invalid prices are rejected.
2. `quantity` is a positive, genuine `int` (`bool` must not count as an integer here).
3. `total` is computed on demand and cached in the instance dictionary.
4. A **successful changed** field value invalidates the cached total and appends an event.
5. An invalid assignment, an unchanged assignment, and a forbidden deletion do not change the cache or event log.
6. Two cart objects never share price, quantity, cached totals, or event logs.

These invariants are a stronger test than merely checking that a descriptor can print a message.

### Step 3 — Define a descriptor that validates before committing

In [78]:
class P20CartField:
    def __init__(self, kind):
        if kind not in {'price', 'quantity'}:
            raise ValueError('unsupported kind')
        self.kind = kind

    def __set_name__(self, owner, name):
        self.name = name
        self.key = '_' + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} is not initialized') from None

    def _validated(self, value):
        if self.kind == 'quantity':
            if type(value) is not int:
                raise TypeError('quantity must be a genuine int')
            if value <= 0:
                raise ValueError('quantity must be positive')
            return value
        if isinstance(value, bool):
            raise TypeError('price may not be bool')
        try:
            amount = Decimal(str(value))
        except (ValueError, ArithmeticError) as exc:
            raise ValueError('invalid decimal price') from exc
        if not amount.is_finite() or amount < 0:
            raise ValueError('price must be finite and nonnegative')
        pennies = Decimal('0.01')
        rounded = amount.quantize(pennies)
        if amount != rounded:
            raise ValueError('price may have at most two decimal places')
        return rounded

    def __set__(self, instance, value):
        new_value = self._validated(value)
        old_value = instance.__dict__.get(self.key)
        if self.key in instance.__dict__ and old_value == new_value:
            return
        instance.__dict__[self.key] = new_value
        instance.__dict__.pop('total', None)
        instance.events.append((self.name, old_value, new_value))

    def __delete__(self, instance):
        raise AttributeError(f'cannot delete required field {self.name}')

**What just happened?** The validator converts before committing and rejects bool explicitly. `Decimal(str(value))` is sufficient for this educational model but is not a universal financial-input parsing interface; real applications must define accepted types, precision, locale, and rounding policies.

### Step 4 — Create an owner class with a cached derived quantity

In [79]:
class P20Cart:
    price = P20CartField('price')
    quantity = P20CartField('quantity')

    def __init__(self, price, quantity):
        self.events = []
        self.total_computations = 0
        self.price = price
        self.quantity = quantity

    @cached_property
    def total(self):
        self.total_computations += 1
        return self.price * self.quantity

p20_cart = P20Cart('2.50', 3)
print('price:', p20_cart.price, 'quantity:', p20_cart.quantity)
print('first total:', p20_cart.total)
print('second total:', p20_cart.total)
print('computations:', p20_cart.total_computations)

price: 2.50 quantity: 3
first total: 7.50
second total: 7.50
computations: 1


**What just happened?** The descriptors are shared *definitions*, while each instance stores its own state. `cached_property` places `total` in that instance’s dictionary after the first calculation.

### Step 5 — Test rejected assignments against an immutable snapshot

In [80]:
p20_state = (p20_cart.price, p20_cart.quantity,
             p20_cart.__dict__['total'], tuple(p20_cart.events),
             p20_cart.total_computations)
for error_type, attr, bad in [
    (ValueError, 'price', '-0.10'),
    (ValueError, 'price', '1.234'),
    (ValueError, 'price', 'NaN'),
    (TypeError, 'price', True),
    (ValueError, 'quantity', 0),
    (TypeError, 'quantity', True),
]:
    expect_error(error_type, lambda attr=attr, bad=bad: setattr(p20_cart, attr, bad))

p20_after = (p20_cart.price, p20_cart.quantity,
             p20_cart.__dict__['total'], tuple(p20_cart.events),
             p20_cart.total_computations)
print('state unchanged after 6 failures:', p20_state == p20_after)

state unchanged after 6 failures: True


**What just happened?** Every invalid value fails *inside validation*, before instance storage, invalidation, or event append. The explicit tuple snapshot avoids accidentally comparing a mutable list with itself.

### Step 6 — Test successful mutation and no-op mutation separately

In [81]:
p20_initial_events = len(p20_cart.events)
p20_cart.price = Decimal('2.50')
print('no-op kept cache:', 'total' in p20_cart.__dict__)
print('no-op added events:', len(p20_cart.events) - p20_initial_events)
p20_cart.quantity = 4
print('changed field invalidated cache:', 'total' not in p20_cart.__dict__)
print('recomputed total:', p20_cart.total)
print('computation count:', p20_cart.total_computations)

no-op kept cache: True
no-op added events: 0
changed field invalidated cache: True
recomputed total: 10.00
computation count: 2


**What just happened?** The unchanged value is normalized and compared before invalidation. A genuinely changed value deletes the cache key so the next read recomputes the derived total.

### Step 7 — Prove instance isolation and forbidden deletion

In [82]:
p20_other = P20Cart('10.00', 2)
print('other total:', p20_other.total)
print('first cart total:', p20_cart.total)
print('separate event lists:', p20_other.events is not p20_cart.events)
expect_error(AttributeError, lambda: delattr(p20_cart, 'price'))
print('first cart after failed deletion:', p20_cart.price, p20_cart.total)

other total: 20.00
first cart total: 10.00
separate event lists: True
first cart after failed deletion: 2.50 10.00


**What just happened?** Neither shared descriptor object stores an instance’s price or quantity. The fields, logs, and caches are separate because storage happens in each instance dictionary.

### Solution 20 — Put the reasoning together

**Answer:** a descriptor can enforce input invariants and participate in a small dependency graph when its setter explicitly invalidates dependent caches. This design assumes single-threaded mutations and an ordinary, successful event-list append; a concurrent or persistent application would need stronger transaction boundaries. The main guarantee demonstrated here is: **validation failure never commits a partial update.**

**Verification.** The next cell uses assertions, including an edge case. Assertions are intentionally executable, not merely a printed expected answer.

In [83]:
assert p20_state == p20_after
assert p20_cart.price == Decimal('2.50')
assert p20_cart.quantity == 4
assert p20_cart.total == Decimal('10.00')
assert p20_cart.total_computations == 2
assert p20_other.total == Decimal('20.00')
assert p20_other.total_computations == 1
assert len(p20_cart.events) == 3  # price, quantity, updated quantity
assert len(p20_other.events) == 2
assert P20Cart.price is P20Cart.__dict__['price']
assert P20Cart.quantity is P20Cart.__dict__['quantity']
print('Problem 20: all checks passed')

Problem 20: all checks passed


**Try it yourself:** Re-run the problem after changing one relevant class, descriptor, or instance attribute. Before each run, write down which object should be returned and why. The assertions above provide a known-good starting point.

---

# Synthesis: the mental model behind all 20 problems

When something surprising happens, use this four-question diagnostic:

1. **Where is the selected attribute stored?** Instance dictionary? First class in the MRO? Metaclass? Raw dictionary inspection and `inspect.getattr_static` can help.
2. **Is it a descriptor and what kind?** Inspect `type(attribute)` for `__get__`, `__set__`, or `__delete__`. Data descriptors and non-data descriptors have different priorities relative to instance storage.
3. **What is returned by `__get__`?** A function, a bound method, a generator function, a coroutine function, a computed value, or the descriptor itself? Check `__func__`, `__self__`, and `inspect` where appropriate.
4. **Where is the state retained?** An instance `__dict__`, slots, a class-level cache, a descriptor-owned weak mapping, or a captured bound method? This determines invalidation, aliasing, and object lifetime.

A descriptor protocol implementation is not a complete application design: access semantics, validation, cache consistency, concurrency, introspection, memory management, and error propagation must be considered separately.

## Extra independent practice (solutions immediately below)

Try answering each question without running new code first:

1. Why does a saved bound method not follow a later class patch?
2. How can a descriptor intercept access to a class object?
3. Can `__getattr__` run even when a descriptor with that name is present?
4. Can a subclass ordinary attribute replace a parent data descriptor?
5. How can you reuse an inherited property getter but replace its setter?
6. Does putting a function in an instance dictionary bind it automatically?
7. Which class does an inherited `classmethod` receive when accessed through a subclass?
8. Which instance attribute invalidates a `cached_property` result?
9. Why can method-level `lru_cache` affect object lifetime?
10. Which slot enables weak references on an otherwise slot-only class?
11. What does `inspect.getattr_static` deliberately avoid?
12. What protocol flag helps `ABCMeta` detect an abstract descriptor?
13. Which decorator helps an instrumentation wrapper keep the original function signature?
14. Why should an async wrapper be declared `async def`?
15. Why must a generator tracer avoid eagerly building a `list`?
16. Which argument does `singledispatchmethod` inspect?
17. What should happen to cache and audit events when validation fails?

### Extra practice: worked reasoning

1. A bound method already stores its function and receiver; a class patch affects fresh attribute lookups.
2. Install the descriptor on a metaclass; classes are instances of their metaclasses.
3. Yes. An `AttributeError` during normal descriptor retrieval permits `__getattr__` fallback.
4. Yes. MRO chooses the class attribute first; descriptor priority is then evaluated for that chosen attribute.
5. Use `@Base.field.setter` and, when appropriate, call `Base.field.fset(self, value)` explicitly.
6. No. Assign `types.MethodType(function, instance)` for intentional instance-specific binding.
7. It receives the subclass through which the classmethod is accessed.
8. The cached property’s own key in that instance’s dictionary; delete it to allow recomputation.
9. Its cache keys include `self` and can retain instances until eviction or cache clearing.
10. `__weakref__`, unless inherited weak-reference support is already available.
11. It avoids executing descriptors and user-defined dynamic attribute lookup.
12. `__isabstractmethod__` on the descriptor object.
13. `functools.wraps`, which installs metadata including `__wrapped__`.
14. To preserve coroutine-function introspection and correctly await the delegate.
15. Eager materialization defeats streaming and changes when work and exceptions happen.
16. The first non-`self`/non-`cls` argument.
17. Both should remain unchanged. Commit only after validation succeeds, and invalidate dependent caches only after an actual change.

## Reference pointers and caveats

- Original source provided by the user: **Functions and Descriptors** (functions implement the non-data descriptor protocol, bound method inspection, and a `MyFunc` descriptor using `types.MethodType`). The new exercises deliberately extend beyond those demonstrations.
- Standard library documentation topics to consult offline or later: the Python descriptor HOWTO, data model (`object.__getattribute__`), `functools.cached_property`, `functools.lru_cache`, `functools.singledispatchmethod`, `inspect.getattr_static`, `abc`, and `weakref.WeakMethod`.
- These examples illustrate mechanisms. Their logging, currency parsing, and storage policies are intentionally scoped rather than advertised as thread-safe or production-ready frameworks.

**You have finished when:** you can predict binding, attribute resolution, storage, and exception behavior *before* executing each experiment, then explain why the assertions pass.